# Missing values: mechanisms and imputation

This notebook is the second lecture of block 1. Missing values are the most common data quality problem, and how to handle them depends on *why* they are missing. We introduce the three missing-data mechanisms (MCAR, MAR, MNAR), compare simple, KNN and iterative imputation and add missing-value indicators. The statistical theory belongs to the statistics module; here we choose, run and interpret the methods. The examples use the second course dataset, the Berlin listings of Inside Airbnb (snapshot of 26 June 2026, CC BY 4.0; prepare it with `uv run python case-study/prepare_airbnb.py`), because missing prices and missing ratings are real and consequential there. The worked example asks whether a missing price is related to the number of reviews, and how well imputation recovers hidden values ([workbook 04](04-case-study-airbnb-missing-and-outliers.ipynb), sections 1–3). Outliers in a single variable follow in block 2 ([05-lecture-outliers-transformations-pipeline.ipynb](05-lecture-outliers-transformations-pipeline.ipynb)).

```mermaid
flowchart TD
    M["Column with missing values"] --> P["Profile: share missing,<br/>indicator vs other columns"]
    P --> Q{"Does missingness depend<br/>on observed columns?"}
    Q -->|"no evidence"| MCAR["Plausibly MCAR:<br/>complete cases unbiased,<br/>simple imputation acceptable"]
    Q -->|yes| MAR["Not MCAR: MAR or MNAR<br/>use model-based imputation<br/>with those columns + indicator"]
    MAR --> S["Sensitivity analysis<br/>for MNAR"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Missing data mechanisms (MCAR, MAR, MNAR)

### Concept

Rubin (1976) classified the reasons for missing values into three **mechanisms**:

- **MCAR (missing completely at random):** whether a value is missing is unrelated to any variable, observed or not, like a coin flip. The rows with a value are a random subsample of all rows.
- **MAR (missing at random):** missingness depends only on variables we observe. Example: younger respondents skip the income question more often, and we know their age.
- **MNAR (missing not at random):** missingness depends on the missing value itself, or on something we do not observe. Example: people with high incomes skip the income question.

The consequences differ:

| Mechanism | Complete-case analysis (drop incomplete rows) | What helps |
|---|---|---|
| MCAR | unbiased, only fewer rows | any reasonable method |
| MAR | biased | imputation or weighting that uses the observed variables |
| MNAR | biased | assumptions or additional data; sensitivity analysis |

The data alone cannot prove MCAR or MAR. They can, however, show that MCAR is **implausible**: if the share of missing values differs between groups of an observed variable, missingness is not completely random.

A small example by hand. Five people, income missing for two:

| age | income |
|---|---|
| 25 | ? |
| 30 | ? |
| 45 | 4,000 |
| 50 | 4,200 |
| 60 | 4,500 |

The two missing values belong to the youngest people. If income rises with age, the mean of the observed incomes (4,233) overestimates the true mean: complete-case analysis is biased. Because age is observed, the pattern is consistent with MAR, and a model of income given age can correct the bias.

### Why it matters

The mechanism decides which treatment is valid. Dropping rows or filling in the mean is harmless under MCAR and misleading under MAR and MNAR, and in real data MCAR is the exception. Many published analyses report results on complete cases without asking who is missing.

### How it works in Python

A simulation with a known truth makes the three mechanisms visible:

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(1)
n = 10_000
age = rng.normal(45, 12, n)
income = 1_000 + 60 * age + rng.normal(0, 500, n)            # monthly income rises with age
sim = pd.DataFrame({"age": age, "income": income})
print(round(sim["income"].mean()))                            # 3686: the true mean

mcar = rng.random(n) < 0.3                                    # a coin flip for every row
mar = rng.random(n) < np.where(age < 40, 0.6, 0.1)            # depends on the observed age
mnar = rng.random(n) < np.where(income > 4000, 0.6, 0.1)      # depends on income itself

for name, miss in [("MCAR", mcar), ("MAR", mar), ("MNAR", mnar)]:
    observed = sim.loc[~miss]
    print(name, round(miss.mean(), 2), round(observed["income"].mean()), round(observed["age"].mean(), 1))
# MCAR 0.3  3681 44.8   <- unbiased, only fewer rows
# MAR  0.27 3877 47.9   <- biased, but the observed ages show why
# MNAR 0.28 3459 42.4   <- biased, and nothing observed explains it fully

3686
MCAR 0.3 3681 44.8
MAR 0.27 3877 47.9
MNAR 0.28 3459 42.4


On the Airbnb listings, a **missingness indicator** (1 = missing) compared across groups shows that missing prices are not MCAR:

In [3]:
import pandas as pd

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
listings["price_missing"] = listings["price"].isna()
print(round(listings["price_missing"].mean(), 3))                              # 0.339
recent = pd.cut(listings["number_of_reviews_ltm"], [-1, 0, 2, 10, 10_000], labels=["0", "1-2", "3-10", ">10"])
print(listings.groupby(recent, observed=True)["price_missing"].mean().round(3).to_dict())
# {'0': 0.601, '1-2': 0.218, '3-10': 0.16, '>10': 0.062}   <- reviews in the last 12 months

blocked = listings["availability_365"].eq(0)                                   # no free night in the next year
print(listings.groupby(blocked)["price_missing"].mean().round(3).to_dict())    # {False: 0.091, True: 0.999}
active = listings[~blocked]
print(active.groupby(active["number_of_reviews_ltm"].eq(0))["price_missing"].mean().round(3).to_dict())
# {False: 0.042, True: 0.2}   <- even among bookable listings, no recent reviews -> more missing prices

0.339
{'0': 0.601, '1-2': 0.218, '3-10': 0.16, '>10': 0.062}
{False: 0.091, True: 0.999}
{False: 0.042, True: 0.2}


![Share of listings without a price by recent reviews and availability, and the structural missingness of the review score and the price](../figures/missingness-pattern.png)

A third of the listings have no price, and listings without a review in the last twelve months lack one ten times as often as busy listings. Taken alone, this suggests "missing prices go with few reviews". The left panel shows the third variable behind it. Inside Airbnb reads the price from the listing page for a bookable date; a listing with **no free night** in the next year shows no price at all (99.9 % missing), and such listings also collect few reviews. Among listings that can be booked, the gap remains (20 % against 4 %): dormant listings often show no price. Availability and recent reviews are observed, so MAR given these variables is a reasonable working assumption; whether a dormant host would charge more or less than others (MNAR) cannot be decided from these data. The right panel shows the extreme case, **structural missingness**: the review score is missing exactly for the 2,573 listings without any review. There is no rating to recover; "no reviews yet" is information in itself. The same holds for the price of a blocked listing.

### In practice

- **Clinical trials**: patients who drop out because of side effects or lack of effect create MNAR data; the European Medicines Agency's *Guideline on missing data in confirmatory clinical trials* (2010) requires sensitivity analyses for this reason.
- **Household surveys**: income is among the items with the highest non-response; the US Census Bureau imputes missing income in the Current Population Survey with a *hot-deck* method that copies values from similar respondents.
- **Electronic health records**: a laboratory value is missing because the test was not ordered, and tests are ordered for patients the physician considers at risk; the fact that a value is missing is itself informative.

> **Caution.** "The data are missing at random" is an assumption, not a finding. You can show evidence *against* MCAR, but you cannot test MAR against MNAR with the observed data. State the assumption and, for important conclusions, check how results change under a plausible MNAR scenario.

## Simple, KNN and iterative imputation

### Concept

**Imputation** replaces missing values by estimates, so that methods that need complete data can run. scikit-learn offers three families:

- **`SimpleImputer`** fills each column with one number: the mean, the median, the most frequent value or a constant. Fast, but it ignores all other columns and shrinks the variance: every imputed row gets the same value.
- **`KNNImputer`** fills a value with the average of the *k* most similar rows (the **nearest neighbours**), measured on the columns that are observed. Variables must be on comparable scales, so standardise them first.
- **`IterativeImputer`** models each column with missing values as a regression on the other columns, fills the gaps with the predictions, and cycles through the columns until the estimates stabilise. This is the idea of **MICE** (multiple imputation by chained equations, van Buuren).

**Single** imputation produces one completed dataset and treats imputed values as if they were measured, which understates uncertainty. **Multiple imputation** creates several completed datasets with random variation, analyses each and combines the results with Rubin's rules; it is the standard for statistical inference (available in `statsmodels.imputation.mice`).

```mermaid
flowchart LR
    S["SimpleImputer<br/>mean / median / constant"] -->|"uses other columns? no"| R1["fast, biased under MAR,<br/>variance shrinks"]
    K["KNNImputer<br/>average of k neighbours"] -->|"uses other columns? yes"| R2["local, needs scaling,<br/>slow on large data"]
    I["IterativeImputer<br/>regression per column"] -->|"uses other columns? yes"| R3["models relations,<br/>basis of MICE"]
```

### Why it matters

Most scikit-learn models refuse missing values. Dropping incomplete rows throws away data and, under MAR, introduces bias. An imputer that uses the related variables can remove much of that bias, but only if the related variables carry information about the missing ones.

### How it works in Python

On the MAR simulation from above, where income depends on the observed age:

In [4]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer

rng = np.random.default_rng(1)
n = 10_000
age = rng.normal(45, 12, n)
income = 1_000 + 60 * age + rng.normal(0, 500, n)
rng.random(n)                                                # keep the random stream as above (MCAR draw)
mar = rng.random(n) < np.where(age < 40, 0.6, 0.1)
X = pd.DataFrame({"age": age, "income": income})
X.loc[mar, "income"] = np.nan

for imputer in [SimpleImputer(strategy="median"), KNNImputer(n_neighbors=10),
                IterativeImputer(random_state=0)]:
    filled = imputer.fit_transform(X)[:, 1]
    rmse = np.sqrt(np.mean((filled[mar] - income[mar]) ** 2))
    print(f"{type(imputer).__name__:17s} mean {filled.mean():.0f}  RMSE {rmse:.0f}")
# SimpleImputer     mean 3877  RMSE 1079   <- mean still biased, large error
# KNNImputer        mean 3691  RMSE 521
# IterativeImputer  mean 3690  RMSE 496    <- uses age; recovers the true mean (3686)

SimpleImputer     mean 3877  RMSE 1079


KNNImputer        mean 3691  RMSE 521
IterativeImputer  mean 3690  RMSE 496


On the Airbnb listings, `bedrooms` is missing for 27 % of the listings. Workbook 04 takes the 8,081 entire homes with a known number of bedrooms, hides 30 % of the values and imputes them from the number of guests and the distance to the centre. The number of guests is strongly related to the number of bedrooms (r = 0.73), and the imputers that use it roughly halve the error: RMSE 0.87 bedrooms for the mean, 1.02 for the median, 0.62 for KNN, 0.59 for the iterative imputer, and 0.64 for a one-line domain rule (median bedrooms of homes for the same number of guests), which is easier to explain than the iterative imputer and almost as good. Imputation works here because a related variable carries the information; it cannot create information that the other variables do not contain.

### In practice

- **Official statistics**: census and survey offices impute item non-response, for example with hot-deck methods, and document the share of imputed values in their quality reports.
- **Epidemiology**: multiple imputation with chained equations is a standard way to handle missing covariates in cohort studies; Sterne et al. (2009, *BMJ*) give widely used guidance on reporting it.
- **Machine learning in production**: the imputer is fitted on the training data and stored with the model, so that new data are filled with the same values (Session 9 builds such pipelines).

> **Warning.** Fit the imputer on the training data only, then apply it to validation and test data. Fitting it on all rows lets the test rows influence the imputed values, a form of data leakage (Session 7).

> **Caution.** Never impute the **target** variable of a model and then evaluate on it. And do not impute a value that is missing *by definition* (a review score for a listing without reviews, the price of a listing with no bookable night, a "date of death" for a living patient); use a category or an indicator instead.

> **Tip.** **Keep track of what was imputed.** Imputation hides the fact that a value was missing. `SimpleImputer(add_indicator=True)` (also `KNNImputer` and `IterativeImputer`) appends a 0/1 column that is 1 where the value was missing, so a model can still use that information. It helps when missingness is informative: in the listings, a missing rating means no reviews yet, and those listings have much longer minimum stays (median 92 nights against 2).
>
> ```python
> imp = SimpleImputer(strategy="median", add_indicator=True).fit(listings[["bedrooms", "review_scores_rating"]])
> print(imp.get_feature_names_out().tolist())
> # ['bedrooms', 'review_scores_rating', 'missingindicator_bedrooms', 'missingindicator_review_scores_rating']
> ```

> 📖 **Read more:** [extras/04-imputation-methods-compared.ipynb](extras/04-imputation-methods-compared.ipynb) (scikit-learn gallery): imputation by zero, by the mean, with KNN and with the iterative imputer, each with a missing-value indicator, compared by the cross-validated error of a model on two datasets with values removed on purpose.

> 📖 **Read more:** [extras/05-iterative-imputer-variants.ipynb](extras/05-iterative-imputer-variants.ipynb) (scikit-learn gallery): compares `IterativeImputer` with different estimators (Bayesian ridge, random forest, a kernel pipeline, k-nearest neighbours) against simple imputation, scored by the error of a downstream regression on the California housing data.

## Check your understanding

1. In a customer survey, satisfaction is missing more often for customers who cancelled their contract. Which mechanism is plausible, and why can the data not settle it?
2. Why does mean imputation reduce the standard deviation of a variable? Which analyses are affected?
3. When is a missing-value indicator useful even if the imputed value itself is poor?

## Further reading

- van Buuren, S. (2018). *Flexible Imputation of Missing Data* (2nd ed.). Chapman & Hall/CRC. Free online: https://stefvanbuuren.name/fimd/
- scikit-learn developers. *Imputation of missing values* (user guide, section 7.4). https://scikit-learn.org/stable/modules/impute.html
- Rubin, D. B. (1976). Inference and missing data. *Biometrika*, 63(3), 581–592. https://doi.org/10.1093/biomet/63.3.581